# Method 5: Contour Approximation
Uses `cv2.findContours` + `cv2.approxPolyDP` to approximate contour shapes into polygonal line segments.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img_color = cv2.imread('cuboid.png')
img_rgb = cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (5, 5), 1.5)

# Adaptive threshold to binarize
binary = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                               cv2.THRESH_BINARY_INV, 51, 15)

# Clean up small noise
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)

# Find contours
contours, _ = cv2.findContours(binary, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
axes[0].imshow(img_rgb); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(img_rgb)
axes[1].axis('off')

# Approximate each contour with a polygon and draw the edges
min_contour_length = 50
total_segments = 0
for cnt in contours:
    if cv2.arcLength(cnt, True) < min_contour_length:
        continue
    # epsilon controls approximation accuracy — smaller = more points
    epsilon = 0.02 * cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, epsilon, True)
    pts = approx.reshape(-1, 2)
    for k in range(len(pts)):
        x1, y1 = pts[k]
        x2, y2 = pts[(k + 1) % len(pts)]
        axes[1].plot([x1, x2], [y1, y2], color='lime', linewidth=2)
        total_segments += 1

axes[1].set_title(f'Contour Approximation ({total_segments} segments)')
plt.tight_layout()
fig.savefig('result_contour_approx.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total polygon segments: {total_segments}')